## Example 1: Find a free local port

In [ ]:
import importlib.metadata
import socket

import portend

# Example 1: find_available_local_port asks the kernel for an unused port
assert importlib.metadata.version("portend") == "3.2.1"
port = portend.find_available_local_port()
assert 0 < port < 65536
with socket.socket() as s:
    s.bind(("127.0.0.1", port))
print("Example 1 OK:", port)

## Example 2: free() returns at once for an unused port

In [ ]:
import portend

# Example 2: nothing listens on the port, so free() returns immediately
port = portend.find_available_local_port()
portend.free("127.0.0.1", port, timeout=2)
print("Example 2 OK:", port)

## Example 3: occupied() waits for a listener

In [ ]:
import socket
import threading
import time

import portend

# Example 3: a server starts listening a moment later; occupied() returns once it does
port = portend.find_available_local_port()
srv = socket.socket()
def start():
    time.sleep(0.3)
    srv.bind(("127.0.0.1", port)); srv.listen()
threading.Thread(target=start).start()
portend.occupied("127.0.0.1", port, timeout=5)
srv.close()
print("Example 3 OK:", port)

## Example 4: Timeouts in both directions

In [ ]:
import socket

import portend

# Example 4: free() on a busy port and occupied() on an idle one raise portend.Timeout
srv = socket.socket(); srv.bind(("127.0.0.1", 0)); srv.listen()
busy = srv.getsockname()[1]
for call, port in ((portend.free, busy), (portend.occupied, portend.find_available_local_port())):
    try:
        call("127.0.0.1", port, timeout=0.5)
    except portend.Timeout as exc:
        print("  timeout:", exc)
    else:
        raise AssertionError(call.__name__)
srv.close()
print("Example 4 OK")

## Example 5: The Checker and PortNotFree

In [ ]:
import socket

import portend

# Example 5: Checker.assert_free raises PortNotFree immediately (no waiting) when something listens
srv = socket.socket(); srv.bind(("127.0.0.1", 0)); srv.listen()
port = srv.getsockname()[1]
try:
    portend.Checker().assert_free("127.0.0.1", port)
except portend.PortNotFree as exc:
    print("  ", exc)
else:
    raise AssertionError("busy port reported free")
srv.close()
portend.Checker().assert_free("127.0.0.1", port)
print("Example 5 OK")